# LLM Fine-Tuning: SFT, LoRA, QLoRA, DPO, RLHF

## Why Fine-Tune?

| Approach | When to Use |
|----------|-------------|
| Prompting | Quick, no training, limited control |
| RAG | External knowledge, factual accuracy |
| Fine-tuning | Specific style/format/domain, consistent behavior |
| Pretraining | Entirely new domain vocabulary |

## Types of Fine-Tuning

1. **Full Fine-Tuning** update all parameters (expensive)
2. **PEFT (Parameter-Efficient Fine-Tuning)** update small fraction
   - LoRA, QLoRA, Prefix Tuning, Prompt Tuning, IA3
3. **SFT (Supervised Fine-Tuning)** standard next-token prediction on instruction data
4. **DPO (Direct Preference Optimization)** align to preferences without RL
5. **RLHF** Reinforcement Learning from Human Feedback

## LoRA: Low-Rank Adaptation

Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times k}$, LoRA learns a low-rank decomposition:

$$W = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and $r \ll \min(d, k)$.

**Trainable parameters:** $r(d + k)$ instead of $dk$

For $d=k=4096$, $r=16$: reduces from $16.7M$ to $131K$ parameters per layer (~128x reduction).

The forward pass during training:
$$h = W_0 x + \frac{\alpha}{r} BAx$$

where $\alpha$ is a scaling factor (usually $\alpha = r$ or $2r$).

## QLoRA: Quantized LoRA

QLoRA loads the base model in **4-bit NormalFloat (NF4)** quantization and trains LoRA adapters in BF16:

- **NF4**: optimal for normally distributed weights
- **Double quantization**: quantize the quantization constants themselves
- **Paged optimizers**: use CPU memory when GPU OOM

Memory savings: ~65B model fits on 1x 48GB GPU (vs 4x 80GB for full fine-tuning).

## DPO: Direct Preference Optimization

RLHF requires training a separate reward model. DPO solves alignment directly:

$$\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\mathbb{E}_{(x,y_w,y_l)}\left[\log\sigma\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

where $y_w$ = preferred response, $y_l$ = rejected response, $\beta$ controls deviation from reference policy.

In [1]:
# LoRA from scratch to understand the math
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """Linear layer with LoRA adapter"""
    def __init__(self, in_features, out_features, r=4, alpha=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Frozen base weights
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.weight.requires_grad = False  # Freeze!

        # LoRA adapters (trainable)
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)   # A: initialize with small random
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))           # B: initialize with zeros

    def forward(self, x):
        # Base model output (frozen)
        base_out = x @ self.weight.T
        # LoRA delta
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        return base_out + self.scaling * lora_out

    def trainable_params(self):
        return self.r * (self.in_features + self.out_features)

    def total_params(self):
        return self.in_features * self.out_features

# Example
lora = LoRALinear(in_features=768, out_features=768, r=16, alpha=32)
x = torch.randn(2, 768)
out = lora(x)

print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')
print(f'Full params:     {lora.total_params():,}')
print(f'LoRA params:     {lora.trainable_params():,}')
print(f'Reduction:       {lora.total_params() / lora.trainable_params():.1f}x')

Input:  torch.Size([2, 768])
Output: torch.Size([2, 768])
Full params:     589,824
LoRA params:     24,576
Reduction:       24.0x


In [2]:
# Fine-tuning with Unsloth (fastest method)
# pip install unsloth

UNSLOTH_CODE = '''
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Load model in 4-bit (QLoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    dtype=None,       # auto-detect
    load_in_4bit=True,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Dataset Alpaca format
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

ALPACA_TEMPLATE = """Below is an instruction that describes a task.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_sample(example):
    return {"text": ALPACA_TEMPLATE.format(**example)}

dataset = dataset.map(format_sample)

# Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        output_dir="./outputs",
        logging_steps=10,
        save_strategy="epoch",
    ),
)
trainer.train()

# Save and push to Hub
model.save_pretrained("lora_model")
model.push_to_hub("your-username/llama-3-8b-finetuned")
'''
print(UNSLOTH_CODE)


from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Load model in 4-bit (QLoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    dtype=None,       # auto-detect
    load_in_4bit=True,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Dataset Alpaca format
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

ALPACA_TEMPLATE = """Below is an instruction that describes a task.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_sample(example):
    return {"text": ALPACA_TEMPLATE.format(**example)}

dataset = dataset.map(format_sample)

# Train

In [3]:
# DPO Training
DPO_CODE = '''
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

# DPO dataset format
dpo_data = [
    {
        "prompt": "Write a poem about stars.",
        "chosen": "Stars are distant suns that light the night,\nEach one a world of wonder and delight...",
        "rejected": "Stars are things. They are in space. They shine."
    },
    # ... more examples
]
dataset = Dataset.from_list(dpo_data)

# DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # if None, uses frozen copy of model
    args=DPOConfig(
        beta=0.1,           # KL penalty coefficient
        max_length=1024,
        max_prompt_length=512,
        per_device_train_batch_size=2,
        num_train_epochs=1,
        output_dir="./dpo_output",
    ),
    train_dataset=dataset,
    tokenizer=tokenizer,
)
dpo_trainer.train()
'''
print(DPO_CODE)


from trl import DPOTrainer, DPOConfig
from datasets import Dataset

# DPO dataset format
dpo_data = [
    {
        "prompt": "Write a poem about stars.",
        "chosen": "Stars are distant suns that light the night,
Each one a world of wonder and delight...",
        "rejected": "Stars are things. They are in space. They shine."
    },
    # ... more examples
]
dataset = Dataset.from_list(dpo_data)

# DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # if None, uses frozen copy of model
    args=DPOConfig(
        beta=0.1,           # KL penalty coefficient
        max_length=1024,
        max_prompt_length=512,
        per_device_train_batch_size=2,
        num_train_epochs=1,
        output_dir="./dpo_output",
    ),
    train_dataset=dataset,
    tokenizer=tokenizer,
)
dpo_trainer.train()



## Additional Learning Resources

### Papers
- [LoRA paper](https://arxiv.org/abs/2106.09685) Hu et al., 2021
- [QLoRA paper](https://arxiv.org/abs/2305.14314) Dettmers et al., 2023
- [DPO paper](https://arxiv.org/abs/2305.18290) Rafailov et al., 2023
- [InstructGPT / RLHF](https://arxiv.org/abs/2203.02155) Ouyang et al., 2022

### Tools
- [Unsloth GitHub](https://github.com/unslothai/unsloth) 2x faster fine-tuning
- [Hugging Face PEFT](https://huggingface.co/docs/peft/)
- [TRL (Transformer RL)](https://huggingface.co/docs/trl/)
- [Axolotl](https://github.com/OpenAccess-AI-Collective/axolotl)

### Tutorials
- [Hugging Face Fine-tuning Course](https://huggingface.co/learn/nlp-course/chapter3)
- [Deeplearning.ai Fine-tuning LLMs](https://www.deeplearning.ai/short-courses/finetuning-large-language-models/)